# 使用 Langfuse 评估智能体

在本 cookbook 中，我们将学习如何使用 [Langfuse](https://langfuse.com/docs) **监控 [OpenAI agent SDK](https://github.com/openai/openai-agents-python) 的内部步骤（traces）**并**评估其性能**。

本指南涵盖团队用于快速、可靠地将智能体投入生产的**在线**和**离线评估**指标。要了解更多评估策略，请查看这篇[博客文章](https://langfuse.com/blog/2025-03-04-llm-evaluation-101-best-practices-and-challenges)。

**为什么 AI 智能体评估很重要：**
- 在任务失败或产生次优结果时调试问题
- 实时监控成本和性能
- 通过持续反馈提高可靠性和安全性

<br>

<div style="position: relative; padding-top: 69.85769728331177%;">
  <iframe
    src="https://customer-xnej9vqjtgxpafyk.cloudflarestream.com/e335d35a0a762b76055f3e0b977f3380/iframe?muted=true&loop=true&autoplay=true&poster=https%3A%2F%2Fcustomer-xnej9vqjtgxpafyk.cloudflarestream.com%2Fe335d35a0a762b76055f3e0b977f3380%2Fthumbnails%2Fthumbnail.jpg%3Ftime%3D%26height%3D600&controls=false"
    loading="lazy"
    style="border: white; position: absolute; top: 0; left: 0; height: 100%; width: 100%; border-radius: 10px;"
    allow="accelerometer; gyroscope; autoplay; encrypted-media; picture-in-picture;"
    allowfullscreen="true"
  ></iframe>
</div>

## 第 0 步：安装所需的库

下面我们安装 `openai-agents` 库（即 [OpenAI Agents SDK](https://github.com/openai/openai-agents-python)）、`pydantic-ai[logfire]` OpenTelemetry 插桩、`langfuse` 以及 Hugging Face 的 `datasets` 库

In [ ]:
%pip install openai-agents nest_asyncio "pydantic-ai[logfire]" langfuse datasets

## 第 1 步：为智能体添加插桩

在本 notebook 中，我们将使用 [Langfuse](https://langfuse.com/) 来追踪、调试和评估我们的智能体。

**注意：** 如果你使用的是 LlamaIndex 或 LangGraph，可以在[这里](https://langfuse.com/docs/integrations/llama-index/workflows)和[这里](https://langfuse.com/docs/integrations/langchain/example-python-langgraph)找到为它们添加插桩的文档。

In [2]:
import os
import base64

# Get keys for your project from the project settings page: https://cloud.langfuse.com
os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..." 
os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..." 
os.environ["LANGFUSE_HOST"] = "https://cloud.langfuse.com" # 🇪🇺 EU region
# os.environ["LANGFUSE_HOST"] = "https://us.cloud.langfuse.com" # 🇺🇸 US region

# Build Basic Auth header.
LANGFUSE_AUTH = base64.b64encode(
    f"{os.environ.get('LANGFUSE_PUBLIC_KEY')}:{os.environ.get('LANGFUSE_SECRET_KEY')}".encode()
).decode()
 
# Configure OpenTelemetry endpoint & headers
os.environ["OTEL_EXPORTER_OTLP_ENDPOINT"] = os.environ.get("LANGFUSE_HOST") + "/api/public/otel"
os.environ["OTEL_EXPORTER_OTLP_HEADERS"] = f"Authorization=Basic {LANGFUSE_AUTH}"

# Your openai key
os.environ["OPENAI_API_KEY"] = "sk-proj-..."

设置好环境变量后，我们就可以初始化 Langfuse 客户端了。`get_client()` 会使用环境变量中提供的凭据初始化 Langfuse 客户端。

In [ ]:
from langfuse import get_client
 
langfuse = get_client()
 
# Verify connection
if langfuse.auth_check():
    print("Langfuse client is authenticated and ready!")
else:
    print("Authentication failed. Please check your credentials and host.")

Pydantic Logfire 为 OpenAI Agents SDK 提供了插桩功能。我们用它将 traces 发送到 [Langfuse OpenTelemetry 后端](https://langfuse.com/docs/opentelemetry/get-started)。

In [4]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
import logfire

# Configure logfire instrumentation.
logfire.configure(
    service_name='my_agent_service',

    send_to_logfire=False,
)
# This method automatically patches the OpenAI Agents SDK to send logs via OTLP to Langfuse.
logfire.instrument_openai_agents()

## 第 2 步：测试插桩

下面是一个简单的问答智能体。我们运行它来确认插桩是否正常工作。如果一切设置正确，你将在可观测性仪表板中看到日志/spans。

In [8]:
import asyncio
from agents import Agent, Runner

async def main():
    agent = Agent(
        name="Assistant",
        instructions="You are a senior software engineer",
    )

    result = await Runner.run(agent, "Tell me why it is important to evaluate AI agents.")
    print(result.final_output)

loop = asyncio.get_running_loop()
await loop.create_task(main())

langfuse.flush()

13:00:52.784 OpenAI Agents trace: Agent workflow
13:00:52.787   Agent run: 'Assistant'
13:00:52.797     Responses API with 'gpt-4o'
Evaluating AI agents is crucial for several reasons:

1. **Performance Assessment**: It helps determine if the agent meets the desired goals and performs tasks effectively. By evaluating, we can assess accuracy, speed, and overall performance.

2. **Reliability and Consistency**: Regular evaluation ensures that the AI behaves consistently under different conditions and is reliable in production environments.

3. **Bias and Fairness**: Identifying and mitigating biases is essential for fair and ethical AI. Evaluation helps uncover any discriminatory patterns in the agent's behavior.

4. **Safety**: Evaluating AI agents ensures they operate safely and do not cause harm or unintended side effects, especially in critical applications.

5. **User Trust**: Proper evaluation builds trust with users and stakeholders by demonstrating that the AI is effective and al

查看你的 [Langfuse Traces 仪表板](https://cloud.langfuse.com/traces)，确认 spans 和日志已被记录。

Langfuse 中的 trace 示例：

![Langfuse 中的 trace 示例](https://langfuse.com/images/cookbook/integration_openai-agents/first-example-trace.png)

_[该 trace 的链接](https://cloud.langfuse.com/project/cloramnkj0002jz088vzn1ja4/traces/0195948781a9f0d78fd5e067154aa508?timestamp=2025-03-14T12%3A01%3A03.401Z&observation=64bcac3cb82d04e9)_

## 第 3 步：观察并评估一个更复杂的智能体

既然已经确认插桩可以正常工作，我们来尝试一个更复杂的查询，看看高级指标（token 用量、延迟、成本等）是如何被追踪的。

In [9]:
import asyncio
from agents import Agent, Runner, function_tool

# Example function tool.
@function_tool
def get_weather(city: str) -> str:
    return f"The weather in {city} is sunny."

agent = Agent(
    name="Hello world",
    instructions="You are a helpful agent.",
    tools=[get_weather],
)

async def main():
    result = await Runner.run(agent, input="What's the weather in Berlin?")
    print(result.final_output)

loop = asyncio.get_running_loop()
await loop.create_task(main())

13:01:15.351 OpenAI Agents trace: Agent workflow
13:01:15.355   Agent run: 'Hello world'
13:01:15.364     Responses API with 'gpt-4o'
13:01:15.999     Function: get_weather
13:01:16.000     Responses API with 'gpt-4o'
The weather in Berlin is currently sunny.


### Trace 结构

Langfuse 会记录一个包含 **spans** 的 **trace**，每个 span 代表智能体逻辑中的一个步骤。在这里，trace 包含整个智能体运行，以及以下子 span：
- 工具调用（get_weather）
- LLM 调用（使用 'gpt-4o' 的 Responses API）

你可以查看这些内容，准确了解时间花在了哪里、使用了多少 token 等：

![Langfuse 中的 trace 树](https://langfuse.com/images/cookbook/integration_openai-agents/trace-tree.png)

_[该 trace 的链接](https://cloud.langfuse.com/project/cloramnkj0002jz088vzn1ja4/traces/019594b5b9a27c5d497b13be71e7f255?timestamp=2025-03-14T12%3A51%3A32.386Z&display=preview&observation=6374a3c96baf831d)_

## 在线评估

在线评估是指在实时的真实环境中（即在生产环境的实际使用过程中）评估智能体。这包括监控智能体在真实用户交互中的表现，并持续分析结果。

我们在[这里](https://langfuse.com/blog/2025-03-04-llm-evaluation-101-best-practices-and-challenges)整理了一份关于不同评估技术的指南。

### 生产环境中需要追踪的常见指标

1. **成本** — 插桩会捕获 token 用量，你可以为每个 token 设定价格，将其换算为大致成本。
2. **延迟** — 观察完成每个步骤或整个运行所需的时间。
3. **用户反馈** — 用户可以提供直接反馈（点赞/点踩），帮助改进或纠正智能体。
4. **LLM 作为评判者（LLM-as-a-Judge）** — 使用另一个 LLM 近乎实时地评估智能体的输出（例如检查是否有害或是否正确）。

下面展示这些指标的示例。

#### 1. 成本

下面的截图展示了 `gpt-4o` 调用的用量。这有助于发现成本高昂的步骤并优化智能体。

![成本](https://langfuse.com/images/cookbook/integration_openai-agents/gpt-4o-costs.png)

_[该 trace 的链接](https://cloud.langfuse.com/project/cloramnkj0002jz088vzn1ja4/traces/019594b5b9a27c5d497b13be71e7f255?timestamp=2025-03-14T12%3A51%3A32.386Z&display=preview&observation=6374a3c96baf831d)_

#### 2. 延迟

我们还可以看到完成每个步骤所花的时间。在下面的示例中，整个运行耗时 7 秒，你可以按步骤进行拆解。这有助于你找出瓶颈并优化智能体。

![延迟](https://langfuse.com/images/cookbook/integration_openai-agents/openai-agent-latency.png)

_[该 trace 的链接](https://cloud.langfuse.com/project/cloramnkj0002jz088vzn1ja4/traces/019594b5b9a27c5d497b13be71e7f255?timestamp=2025-03-14T12%3A51%3A32.386Z&display=timeline&observation=b12967a01b3f8bcb)_

#### 3. 附加属性

Langfuse 允许你为 spans 传入附加属性，包括 `user_id`、`tags`、`session_id` 以及自定义的 `metadata`。用这些细节丰富 traces，对于分析、调试和监控应用在不同用户或会话中的行为非常重要。

在本示例中，我们将 [user_id](https://langfuse.com/docs/tracing-features/users)、[session_id](https://langfuse.com/docs/tracing-features/sessions) 和 [trace_tags](https://langfuse.com/docs/tracing-features/tags) 传给 Langfuse。 

In [10]:
input_query = "Why is AI agent evaluation important?"

with langfuse.start_as_current_span(
    name="OpenAI-Agent-Trace",
    ) as span:
    
    # Run your application here
    async def main(input_query):
            agent = Agent(
                name = "Assistant",
                instructions = "You are a helpful assistant.",
            )

            result = await Runner.run(agent, input_query)
            print(result.final_output)
            return result

    result = await main(input_query)
 
    # Pass additional attributes to the span
    span.update_trace(
        input=input_query,
        output=result,
        user_id="user_123",
        session_id="my-agent-session",
        tags=["staging", "demo", "OpenAI Agent SDK"],
        metadata={"email": "user@langfuse.com"},
        version="1.0.0"
        )
 
# Flush events in short-lived applications
langfuse.flush()

13:02:41.552 OpenAI Agents trace: Agent workflow
13:02:41.553   Agent run: 'Assistant'
13:02:41.554     Responses API with 'gpt-4o'
AI agent evaluation is crucial for several reasons:

1. **Performance Metrics**: It helps determine how well an AI agent performs its tasks, ensuring it meets the desired standards and objectives.

2. **Reliability and Safety**: Evaluation ensures the agent behaves consistently and safely in different scenarios, reducing risks of unintended consequences.

3. **Bias Detection**: By evaluating AI agents, developers can identify and mitigate biases, ensuring fair and equitable outcomes for all users.

4. **Benchmarking and Comparison**: Evaluation allows for the comparison of different AI models or versions, facilitating improvements and advancements.

5. **User Trust**: Demonstrating the effectiveness and reliability of an AI agent builds trust with users, encouraging adoption and usage.

6. **Regulatory Compliance**: Proper evaluation helps ensure AI system

![Langfuse 中的 trace 示例](https://langfuse.com/images/cookbook/integration_openai-agents/openai-agent-sdk-custom-attributes.png)

#### 4. 用户反馈

如果你的智能体嵌入在用户界面中，就可以记录直接的用户反馈（例如聊天界面中的点赞/点踩）。下面是一个使用 `IPython.display` 实现简单反馈机制的示例。

在下面的代码片段中，当用户发送聊天消息时，我们会捕获 OpenTelemetry trace ID。如果用户喜欢/不喜欢最后一条回答，我们就为该 trace 附加一个分数。

In [38]:
from agents import Agent, Runner, WebSearchTool
from opentelemetry.trace import format_trace_id
import ipywidgets as widgets
from IPython.display import display
from langfuse import get_client
 
langfuse = get_client()

# Define your agent with the web search tool
agent = Agent(
    name="WebSearchAgent",
    instructions="You are an agent that can search the web.",
    tools=[WebSearchTool()]
)

def on_feedback(button):
    if button.icon == "thumbs-up":
      langfuse.create_score(
            value=1,
            name="user-feedback",
            comment="The user gave this response a thumbs up",
            trace_id=trace_id
        )
    elif button.icon == "thumbs-down":
      langfuse.create_score(
            value=0,
            name="user-feedback",
            comment="The user gave this response a thumbs down",
            trace_id=trace_id
        )
    print("Scored the trace in Langfuse")

user_input = input("Enter your question: ")

# Run agent
with langfuse.start_as_current_span(
    name="OpenAI-Agent-Trace",
    ) as span:
    
    # Run your application here
    result = Runner.run_sync(agent, user_input)
    print(result.final_output)

    result = await main(user_input)
    trace_id = langfuse.get_current_trace_id()

    span.update_trace(
        input=user_input,
        output=result.final_output,
    )

# Get feedback
print("How did you like the agent response?")

thumbs_up = widgets.Button(description="👍", icon="thumbs-up")
thumbs_down = widgets.Button(description="👎", icon="thumbs-down")

thumbs_up.on_click(on_feedback)
thumbs_down.on_click(on_feedback)

display(widgets.HBox([thumbs_up, thumbs_down]))

# Flush events in short-lived applications
langfuse.flush()

Enter your question: What is Langfuse?
13:54:41.574 OpenAI Agents trace: Agent workflow
13:54:41.575   Agent run: 'WebSearchAgent'
13:54:41.577     Responses API with 'gpt-4o'
Langfuse is an open-source engineering platform designed to enhance the development, monitoring, and optimization of Large Language Model (LLM) applications. It offers a suite of tools that provide observability, prompt management, evaluations, and metrics, facilitating the debugging and improvement of LLM-based solutions. ([toolkitly.com](https://www.toolkitly.com/langfuse?utm_source=openai))

**Key Features of Langfuse:**

- **LLM Observability:** Langfuse enables developers to monitor and analyze the performance of language models by tracking API calls, user inputs, prompts, and outputs. This observability aids in understanding model behavior and identifying areas for improvement. ([toolkitly.com](https://www.toolkitly.com/langfuse?utm_source=openai))

- **Prompt Management:** The platform provides tools for m

Scored the trace in Langfuse


随后用户反馈会被记录到 Langfuse 中：

![用户反馈正在被记录到 Langfuse 中](https://langfuse.com/images/cookbook/integration_openai-agents/open-ai-agent-user-feedback.png)

#### 5. LLM 作为评判者

LLM 作为评判者（LLM-as-a-Judge）是自动评估智能体输出的另一种方式。你可以设置一个单独的 LLM 调用，来衡量输出的正确性、有害性、风格或任何你关心的其他标准。

**工作流程**：
1. 定义一个**评估模板**，例如 "Check if the text is toxic."
2. 设置一个用作评判模型的模型；本例中为 `gpt-4o-mini`。
2. 每当智能体生成输出时，将该输出连同模板一起传给“评判” LLM。
3. 评判 LLM 返回一个评分或标签，你将其记录到可观测性工具中。

Langfuse 中的示例：

![LLM-as-a-Judge 评估模板](https://langfuse.com/images/cookbook/integration_openai-agents/evaluator-template.png)
![LLM-as-a-Judge 评估器](https://langfuse.com/images/cookbook/integration_openai-agents/evaluator.png)

In [40]:
# Example: Checking if the agent’s output is toxic or not.
from agents import Agent, Runner, WebSearchTool

# Define your agent with the web search tool
agent = Agent(
    name="WebSearchAgent",
    instructions="You are an agent that can search the web.",
    tools=[WebSearchTool()]
)

input_query = "Is eating carrots good for the eyes?"

# Run agent
with langfuse.start_as_current_span(name="OpenAI-Agent-Trace") as span:
    # Run your agent with a query
    result = Runner.run_sync(agent, input_query)

    # Add input and output values to parent trace
    span.update_trace(
        input=input_query,
        output=result.final_output,
    )

14:05:34.735 OpenAI Agents trace: Agent workflow
14:05:34.736   Agent run: 'WebSearchAgent'
14:05:34.738     Responses API with 'gpt-4o'


可以看到，本示例的回答被评判为 "not toxic"（无害）。

![LLM-as-a-Judge 评估分数](https://langfuse.com/images/cookbook/integration_openai-agents/llm-as-a-judge-score.png)

#### 6. 可观测性指标概览

所有这些指标都可以在仪表板中一起可视化。这样你就能快速了解智能体在大量会话中的表现，并随时间追踪质量指标。

![可观测性指标概览](https://langfuse.com/images/cookbook/integration_openai-agents/dashboard-dark.png)

## 离线评估

在线评估对于实时反馈至关重要，但你还需要**离线评估**——在开发之前或期间进行系统性检查。这有助于在将变更推向生产环境之前保持质量和可靠性。

### 数据集评估

在离线评估中，你通常需要：
1. 准备一个基准数据集（包含提示与预期输出的配对）
2. 在该数据集上运行智能体
3. 将输出与预期结果进行比较，或使用额外的评分机制

下面我们使用 [search-dataset](https://huggingface.co/datasets/junzhang1207/search-dataset) 演示这种方法，该数据集包含可通过网页搜索工具回答的问题以及预期答案。

In [44]:
import pandas as pd
from datasets import load_dataset

# Fetch search-dataset from Hugging Face
dataset = load_dataset("junzhang1207/search-dataset", split = "train")
df = pd.DataFrame(dataset)
print("First few rows of search-dataset:")
print(df.head())

README.md:   0%|          | 0.00/2.12k [00:00<?, ?B/s]

data-samples.json:   0%|          | 0.00/2.48k [00:00<?, ?B/s]

data.jsonl:   0%|          | 0.00/316k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/934 [00:00<?, ? examples/s]

First few rows of GSM8K dataset:
                                     id  \
0  20caf138-0c81-4ef9-be60-fe919e0d68d4   
1  1f37d9fd-1bcc-4f79-b004-bc0e1e944033   
2  76173a7f-d645-4e3e-8e0d-cca139e00ebe   
3  5f5ef4ca-91fe-4610-a8a9-e15b12e3c803   
4  64dbed0d-d91b-4acd-9a9c-0a7aa83115ec   

                                            question  \
0                 steve jobs statue location budapst   
1  Why is the Battle of Stalingrad considered a t...   
2  In what year did 'The Birth of a Nation' surpa...   
3  How many Russian soldiers surrendered to AFU i...   
4   What event led to the creation of Google Images?   

                                     expected_answer       category       area  
0  The Steve Jobs statue is located in Budapest, ...           Arts  Knowledge  
1  The Battle of Stalingrad is considered a turni...   General News       News  
2  This question is based on a false premise. 'Th...  Entertainment       News  
3  About 300 Russian soldiers surrendered to th

接下来，我们在 Langfuse 中创建一个数据集实体来追踪各次运行。然后，将数据集中的每一项添加到系统中。

In [52]:
from langfuse import get_client
langfuse = get_client()

langfuse_dataset_name = "search-dataset_huggingface_openai-agent"

# Create a dataset in Langfuse
langfuse.create_dataset(
    name=langfuse_dataset_name,
    description="search-dataset uploaded from Huggingface",
    metadata={
        "date": "2025-03-14",
        "type": "benchmark"
    }
)

Dataset(id='cm88w66t102qpad07xhgeyaej', name='search-dataset_huggingface_openai-agent', description='search-dataset uploaded from Huggingface', metadata={'date': '2025-03-14', 'type': 'benchmark'}, project_id='cloramnkj0002jz088vzn1ja4', created_at=datetime.datetime(2025, 3, 14, 14, 47, 14, 676000, tzinfo=datetime.timezone.utc), updated_at=datetime.datetime(2025, 3, 14, 14, 47, 14, 676000, tzinfo=datetime.timezone.utc))

In [53]:
for idx, row in df.iterrows():
    langfuse.create_dataset_item(
        dataset_name=langfuse_dataset_name,
        input={"text": row["question"]},
        expected_output={"text": row["expected_answer"]}
    )
    if idx >= 49: # For this example, we upload only the first 50 items
        break

![Langfuse 中的数据集条目](https://langfuse.com/images/cookbook/integration_openai-agents/example-dataset.png)

#### 在数据集上运行智能体

我们定义一个辅助函数 `run_openai_agent()`，它会：
1. 启动一个 Langfuse span
2. 针对提示运行智能体
3. 在 Langfuse 中记录 trace ID

然后，我们遍历每个数据集条目，运行智能体，并将 trace 关联到该数据集条目。如有需要，还可以附加一个快速评估分数。

In [ ]:
from agents import Agent, Runner, WebSearchTool
from langfuse import get_client
 
langfuse = get_client()
dataset_name = "search-dataset_huggingface_openai-agent"
current_run_name = "qna_model_v3_run_05_20" # Identifies this specific evaluation run

agent = Agent(
    name="WebSearchAgent",
    instructions="You are an agent that can search the web.",
    tools=[WebSearchTool(search_context_size= "high")]
)
 
# Assume 'run_openai_agent' is your instrumented application function
def run_openai_agent(question):
    with langfuse.start_as_current_generation(name="qna-llm-call") as generation:
        # Simulate LLM call
        result = Runner.run_sync(agent, question)
 
        # Update the trace with the input and output
        generation.update_trace(
            input= question,
            output=result.final_output,
        )

        return result.final_output
 
dataset = langfuse.get_dataset(name=dataset_name) # Fetch your pre-populated dataset
 
for item in dataset.items:
 
    # Use the item.run() context manager
    with item.run(
        run_name=current_run_name,
        run_metadata={"model_provider": "OpenAI", "temperature_setting": 0.7},
        run_description="Evaluation run for Q&A model v3 on May 20th"
    ) as root_span: # root_span is the root span of the new trace for this item and run.
        # All subsequent langfuse operations within this block are part of this trace.
 
        # Call your application logic
        generated_answer = run_openai_agent(question=item.input["text"])

        print(item.input)

你可以使用不同的以下配置重复这个过程：
- 搜索工具（例如为 OpenAI 的 `WebSearchTool` 设置不同的上下文大小）
- 模型（gpt-4o-mini、o1 等）
- 工具（使用搜索 vs. 不使用搜索）

然后在 Langfuse 中将它们并排比较。在本示例中，我在 50 个数据集问题上运行了 3 次智能体。每次运行都为 OpenAI 的 `WebSearchTool` 的上下文大小使用了不同的设置。可以看到，增大上下文大小也使回答正确率从 `0.89` 小幅提升到 `0.92`。`correct_answer` 分数由一个 [LLM-as-a-Judge 评估器](https://langfuse.com/docs/scores/model-based-evals) 生成，该评估器被设置为根据数据集中给出的示例答案来评判回答的正确性。

![数据集运行概览](https://langfuse.com/images/cookbook/integration_openai-agents/dataset_runs.png)
![数据集运行对比](https://langfuse.com/images/cookbook/integration_openai-agents/dataset-run-comparison.png)